In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!sudo apt-get update -q
!sudo apt-get install bcftools -y -q
!pip install cyvcf2 --quiet

import os
import subprocess
from cyvcf2 import VCF

INPUT_DIR = '/content/drive/MyDrive/FYP_DATA/DATA/processed_data/SG10K/sas_filtered'
OUTPUT_DIR = '/content/drive/MyDrive/FYP_DATA/DATA/processed_data/SG10K/sas_filtered_normalized'
REFERENCE_FASTA = '/content/drive/MyDrive/FYP_DATA/DATA/raw_data/reference_fasta/hg38_dna.fa'

# ===================CHANGE THE CHROMOSOME NUMBER ON EVERY RUN ==========================
chr_num = "3"
# =======================================================================================

input_vcf = os.path.join(INPUT_DIR, f'chr{chr_num}_sas_filtered.vcf.gz')

output_vcf = os.path.join(OUTPUT_DIR, f'chr{chr_num}_sas_normalized.vcf.gz')

temp_chr_renamed = os.path.join(OUTPUT_DIR, f'temp_chr{chr_num}_renamed.vcf.gz')
print("=" * 80)
print(f"SG10K NORMALIZATION PIPELINE - CHR{chr_num}")
print("=" * 80)
print(f"Input:  {input_vcf}")
print(f"Output: {output_vcf}")
print(f"Reference: {REFERENCE_FASTA}")

# ============================================================
# STEP 1: Add "chr" prefix to chromosome names
# ============================================================
print("\n" + "=" * 80)
print("STEP 1/4: Adding 'chr' prefix to chromosome names...")
print("=" * 80)

# Create chromosome rename file (only chr1-22 for SG10K)
chr_rename_file = '/tmp/chr_rename_sg10k.txt'
with open(chr_rename_file, 'w') as f:
    for i in range(1, 23):
        f.write(f"{i} chr{i}\n")

print(f"📝 Renaming: {chr_num} → chr{chr_num}")

rename_cmd = f"""
bcftools annotate \
    --rename-chrs {chr_rename_file} \
    -O z \
    -o {temp_chr_renamed} \
    {input_vcf}
"""

print("⏱️  Renaming chromosomes...")
result = subprocess.run(
    rename_cmd,
    shell=True,
    capture_output=True,
    text=True
)

if result.returncode != 0:
    print("\n❌ CHROMOSOME RENAMING FAILED!")
    print(result.stderr)
    raise Exception("Chromosome renaming failed")

print("✓ Chromosomes renamed")

# ============================================================
# STEP 2: Index renamed VCF
# ============================================================
print("\n" + "=" * 80)
print("STEP 2/4: Indexing renamed VCF...")
print("=" * 80)

print("   Creating index for renamed file...")
index_result = subprocess.run(
    f"bcftools index -t -f {temp_chr_renamed}",
    shell=True,
    capture_output=True,
    text=True
)

if index_result.returncode != 0:
    print(f"   ❌ Indexing failed!")
    print(f"   Error: {index_result.stderr}")
    raise Exception("Failed to create index")
else:
    print(f"   ✓ Index created")

# Verify index exists
if os.path.exists(f"{temp_chr_renamed}.tbi"):
    index_size = os.path.getsize(f"{temp_chr_renamed}.tbi") / 1024
    print(f"   ✓ Index file verified ({index_size:.1f} KB)")
else:
    print("   ❌ Index file not found!")
    raise Exception("Failed to create index")


# ============================================================
# STEP 3: Normalize VCF
# ============================================================
print("\n" + "=" * 80)
print("STEP 3/4: Normalizing VCF...")
print("=" * 80)
print("📋 Operations:")
print("   - Left-align indels")
print("   - Split multi-allelic variants (-m-)")
print("   - Check reference alleles")

norm_cmd = f"""
bcftools norm \
    -m- \
    -f {REFERENCE_FASTA} \
    --check-ref w \
    -O z \
    -o {output_vcf} \
    {temp_chr_renamed}
"""

print("\n⏱️  Normalizing (this may take several minutes)...")
result = subprocess.run(
    norm_cmd,
    shell=True,
    capture_output=True,
    text=True
)

# ============================================================
# STEP 4: Index normalized VCF
# ============================================================
print("\n" + "=" * 80)
print("STEP 4/4: Indexing normalized VCF...")
print("=" * 80)

index_cmd = f"bcftools index -t -f {output_vcf}"

result = subprocess.run(
    index_cmd,
    shell=True,
    capture_output=True,
    text=True
)

if result.returncode == 0:
    print(f"✓ Index created: {output_vcf}.tbi")
else:
    print("❌ Indexing failed!")
    print(result.stderr)
    raise Exception("Indexing failed")


# ============================================================
# STEP 5: Cleanup temporary files
# ============================================================
print("\n" + "=" * 80)
print("CLEANUP: Removing temporary files...")
print("=" * 80)

try:
    if os.path.exists(temp_chr_renamed):
        os.remove(temp_chr_renamed)
        print(f"✓ Removed: {temp_chr_renamed}")
    if os.path.exists(f"{temp_chr_renamed}.tbi"):
        os.remove(f"{temp_chr_renamed}.tbi")
        print(f"✓ Removed: {temp_chr_renamed}.tbi")
except Exception as e:
    print(f"⚠️  Cleanup warning: {e}")

print("\n" + "=" * 80)
print("🎉 NORMALIZATION COMPLETE!")
print("=" * 80)
print(f"Final output: {output_vcf}")
print(f"Index file: {output_vcf}.tbi")
print("=" * 80)


Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://cli.github.com/packages stable/main amd64 Packages [356 B]
Hit:3 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:4 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:5 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:6 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:7 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [83.8 kB]
Get:8 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Get:10 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [2,887 kB]
Get:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease [24.6 kB]
Get:12 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:13 http://security.ubuntu.com/ubuntu jammy-security/multiverse amd64 Packages [62.6 kB]
Get:14 http

In [3]:
!sudo apt-get update -q
!sudo apt-get install bcftools -y -q
!pip install cyvcf2 --quiet

import os
import subprocess
from cyvcf2 import VCF

INPUT_DIR = '/content/drive/MyDrive/FYP_DATA/DATA/processed_data/SG10K/sas_filtered'
OUTPUT_DIR = '/content/drive/MyDrive/FYP_DATA/DATA/processed_data/SG10K/sas_filtered_normalized'
REFERENCE_FASTA = '/content/drive/MyDrive/FYP_DATA/DATA/raw_data/reference_fasta/hg38_dna.fa'

# ===================CHANGE THE CHROMOSOME NUMBER ON EVERY RUN ==========================
chr_num = "17"
# =======================================================================================

input_vcf = os.path.join(INPUT_DIR, f'chr{chr_num}_sas_filtered.vcf.gz')

output_vcf = os.path.join(OUTPUT_DIR, f'chr{chr_num}_sas_normalized.vcf.gz')

temp_chr_renamed = os.path.join(OUTPUT_DIR, f'temp_chr{chr_num}_renamed.vcf.gz')
print("=" * 80)
print(f"SG10K NORMALIZATION PIPELINE - CHR{chr_num}")
print("=" * 80)
print(f"Input:  {input_vcf}")
print(f"Output: {output_vcf}")
print(f"Reference: {REFERENCE_FASTA}")

# ============================================================
# STEP 1: Add "chr" prefix to chromosome names
# ============================================================
print("\n" + "=" * 80)
print("STEP 1/4: Adding 'chr' prefix to chromosome names...")
print("=" * 80)

# Create chromosome rename file (only chr1-22 for SG10K)
chr_rename_file = '/tmp/chr_rename_sg10k.txt'
with open(chr_rename_file, 'w') as f:
    for i in range(1, 23):
        f.write(f"{i} chr{i}\n")

print(f"📝 Renaming: {chr_num} → chr{chr_num}")

rename_cmd = f"""
bcftools annotate \
    --rename-chrs {chr_rename_file} \
    -O z \
    -o {temp_chr_renamed} \
    {input_vcf}
"""

print("⏱️  Renaming chromosomes...")
result = subprocess.run(
    rename_cmd,
    shell=True,
    capture_output=True,
    text=True
)

if result.returncode != 0:
    print("\n❌ CHROMOSOME RENAMING FAILED!")
    print(result.stderr)
    raise Exception("Chromosome renaming failed")

print("✓ Chromosomes renamed")

# ============================================================
# STEP 2: Index renamed VCF
# ============================================================
print("\n" + "=" * 80)
print("STEP 2/4: Indexing renamed VCF...")
print("=" * 80)

print("   Creating index for renamed file...")
index_result = subprocess.run(
    f"bcftools index -t -f {temp_chr_renamed}",
    shell=True,
    capture_output=True,
    text=True
)

if index_result.returncode != 0:
    print(f"   ❌ Indexing failed!")
    print(f"   Error: {index_result.stderr}")
    raise Exception("Failed to create index")
else:
    print(f"   ✓ Index created")

# Verify index exists
if os.path.exists(f"{temp_chr_renamed}.tbi"):
    index_size = os.path.getsize(f"{temp_chr_renamed}.tbi") / 1024
    print(f"   ✓ Index file verified ({index_size:.1f} KB)")
else:
    print("   ❌ Index file not found!")
    raise Exception("Failed to create index")


# ============================================================
# STEP 3: Normalize VCF
# ============================================================
print("\n" + "=" * 80)
print("STEP 3/4: Normalizing VCF...")
print("=" * 80)
print("📋 Operations:")
print("   - Left-align indels")
print("   - Split multi-allelic variants (-m-)")
print("   - Check reference alleles")

norm_cmd = f"""
bcftools norm \
    -m- \
    -f {REFERENCE_FASTA} \
    --check-ref w \
    -O z \
    -o {output_vcf} \
    {temp_chr_renamed}
"""

print("\n⏱️  Normalizing (this may take several minutes)...")
result = subprocess.run(
    norm_cmd,
    shell=True,
    capture_output=True,
    text=True
)

# ============================================================
# STEP 4: Index normalized VCF
# ============================================================
print("\n" + "=" * 80)
print("STEP 4/4: Indexing normalized VCF...")
print("=" * 80)

index_cmd = f"bcftools index -t -f {output_vcf}"

result = subprocess.run(
    index_cmd,
    shell=True,
    capture_output=True,
    text=True
)

if result.returncode == 0:
    print(f"✓ Index created: {output_vcf}.tbi")
else:
    print("❌ Indexing failed!")
    print(result.stderr)
    raise Exception("Indexing failed")


# ============================================================
# STEP 5: Cleanup temporary files
# ============================================================
print("\n" + "=" * 80)
print("CLEANUP: Removing temporary files...")
print("=" * 80)

try:
    if os.path.exists(temp_chr_renamed):
        os.remove(temp_chr_renamed)
        print(f"✓ Removed: {temp_chr_renamed}")
    if os.path.exists(f"{temp_chr_renamed}.tbi"):
        os.remove(f"{temp_chr_renamed}.tbi")
        print(f"✓ Removed: {temp_chr_renamed}.tbi")
except Exception as e:
    print(f"⚠️  Cleanup warning: {e}")

print("\n" + "=" * 80)
print("🎉 NORMALIZATION COMPLETE!")
print("=" * 80)
print(f"Final output: {output_vcf}")
print(f"Index file: {output_vcf}.tbi")
print("=" * 80)


Hit:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:2 https://cli.github.com/packages stable InRelease
Hit:3 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:4 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists...
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists...
Building dependency tree...
Reading state information...
bcftools is already the newest version (1.13-1).
0 upgraded, 0 newly installed, 0 to remove and 35 not upgraded.
S

In [4]:
# !sudo apt-get update -q
# !sudo apt-get install bcftools -y -q
# !pip install cyvcf2 --quiet

# import os
# import subprocess
# from cyvcf2 import VCF

INPUT_DIR = '/content/drive/MyDrive/FYP_DATA/DATA/processed_data/SG10K/sas_filtered'
OUTPUT_DIR = '/content/drive/MyDrive/FYP_DATA/DATA/processed_data/SG10K/sas_filtered_normalized'
REFERENCE_FASTA = '/content/drive/MyDrive/FYP_DATA/DATA/raw_data/reference_fasta/hg38_dna.fa'

# ===================CHANGE THE CHROMOSOME NUMBER ON EVERY RUN ==========================
chr_num = "20"
# =======================================================================================

input_vcf = os.path.join(INPUT_DIR, f'chr{chr_num}_sas_filtered.vcf.gz')

output_vcf = os.path.join(OUTPUT_DIR, f'chr{chr_num}_sas_normalized.vcf.gz')

temp_chr_renamed = os.path.join(OUTPUT_DIR, f'temp_chr{chr_num}_renamed.vcf.gz')
print("=" * 80)
print(f"SG10K NORMALIZATION PIPELINE - CHR{chr_num}")
print("=" * 80)
print(f"Input:  {input_vcf}")
print(f"Output: {output_vcf}")
print(f"Reference: {REFERENCE_FASTA}")

# ============================================================
# STEP 1: Add "chr" prefix to chromosome names
# ============================================================
print("\n" + "=" * 80)
print("STEP 1/4: Adding 'chr' prefix to chromosome names...")
print("=" * 80)

# Create chromosome rename file (only chr1-22 for SG10K)
chr_rename_file = '/tmp/chr_rename_sg10k.txt'
with open(chr_rename_file, 'w') as f:
    for i in range(1, 23):
        f.write(f"{i} chr{i}\n")

print(f"📝 Renaming: {chr_num} → chr{chr_num}")

rename_cmd = f"""
bcftools annotate \
    --rename-chrs {chr_rename_file} \
    -O z \
    -o {temp_chr_renamed} \
    {input_vcf}
"""

print("⏱️  Renaming chromosomes...")
result = subprocess.run(
    rename_cmd,
    shell=True,
    capture_output=True,
    text=True
)

if result.returncode != 0:
    print("\n❌ CHROMOSOME RENAMING FAILED!")
    print(result.stderr)
    raise Exception("Chromosome renaming failed")

print("✓ Chromosomes renamed")

# ============================================================
# STEP 2: Index renamed VCF
# ============================================================
print("\n" + "=" * 80)
print("STEP 2/4: Indexing renamed VCF...")
print("=" * 80)

print("   Creating index for renamed file...")
index_result = subprocess.run(
    f"bcftools index -t -f {temp_chr_renamed}",
    shell=True,
    capture_output=True,
    text=True
)

if index_result.returncode != 0:
    print(f"   ❌ Indexing failed!")
    print(f"   Error: {index_result.stderr}")
    raise Exception("Failed to create index")
else:
    print(f"   ✓ Index created")

# Verify index exists
if os.path.exists(f"{temp_chr_renamed}.tbi"):
    index_size = os.path.getsize(f"{temp_chr_renamed}.tbi") / 1024
    print(f"   ✓ Index file verified ({index_size:.1f} KB)")
else:
    print("   ❌ Index file not found!")
    raise Exception("Failed to create index")


# ============================================================
# STEP 3: Normalize VCF
# ============================================================
print("\n" + "=" * 80)
print("STEP 3/4: Normalizing VCF...")
print("=" * 80)
print("📋 Operations:")
print("   - Left-align indels")
print("   - Split multi-allelic variants (-m-)")
print("   - Check reference alleles")

norm_cmd = f"""
bcftools norm \
    -m- \
    -f {REFERENCE_FASTA} \
    --check-ref w \
    -O z \
    -o {output_vcf} \
    {temp_chr_renamed}
"""

print("\n⏱️  Normalizing (this may take several minutes)...")
result = subprocess.run(
    norm_cmd,
    shell=True,
    capture_output=True,
    text=True
)

# ============================================================
# STEP 4: Index normalized VCF
# ============================================================
print("\n" + "=" * 80)
print("STEP 4/4: Indexing normalized VCF...")
print("=" * 80)

index_cmd = f"bcftools index -t -f {output_vcf}"

result = subprocess.run(
    index_cmd,
    shell=True,
    capture_output=True,
    text=True
)

if result.returncode == 0:
    print(f"✓ Index created: {output_vcf}.tbi")
else:
    print("❌ Indexing failed!")
    print(result.stderr)
    raise Exception("Indexing failed")


# ============================================================
# STEP 5: Cleanup temporary files
# ============================================================
print("\n" + "=" * 80)
print("CLEANUP: Removing temporary files...")
print("=" * 80)

try:
    if os.path.exists(temp_chr_renamed):
        os.remove(temp_chr_renamed)
        print(f"✓ Removed: {temp_chr_renamed}")
    if os.path.exists(f"{temp_chr_renamed}.tbi"):
        os.remove(f"{temp_chr_renamed}.tbi")
        print(f"✓ Removed: {temp_chr_renamed}.tbi")
except Exception as e:
    print(f"⚠️  Cleanup warning: {e}")

print("\n" + "=" * 80)
print("🎉 NORMALIZATION COMPLETE!")
print("=" * 80)
print(f"Final output: {output_vcf}")
print(f"Index file: {output_vcf}.tbi")
print("=" * 80)


SG10K NORMALIZATION PIPELINE - CHR20
Input:  /content/drive/MyDrive/FYP_DATA/DATA/processed_data/SG10K/sas_filtered/chr20_sas_filtered.vcf.gz
Output: /content/drive/MyDrive/FYP_DATA/DATA/processed_data/SG10K/sas_filtered_normalized/chr20_sas_normalized.vcf.gz
Reference: /content/drive/MyDrive/FYP_DATA/DATA/raw_data/reference_fasta/hg38_dna.fa

STEP 1/4: Adding 'chr' prefix to chromosome names...
📝 Renaming: 20 → chr20
⏱️  Renaming chromosomes...
✓ Chromosomes renamed

STEP 2/4: Indexing renamed VCF...
   Creating index for renamed file...
   ✓ Index created
   ✓ Index file verified (52.0 KB)

STEP 3/4: Normalizing VCF...
📋 Operations:
   - Left-align indels
   - Split multi-allelic variants (-m-)
   - Check reference alleles

⏱️  Normalizing (this may take several minutes)...

STEP 4/4: Indexing normalized VCF...
✓ Index created: /content/drive/MyDrive/FYP_DATA/DATA/processed_data/SG10K/sas_filtered_normalized/chr20_sas_normalized.vcf.gz.tbi

CLEANUP: Removing temporary files...
✓ Remo